# 🌳 Yggdrasil — Segmentação de LGD

Tutorial do subpacote `yggdrasil.credit_risk.lgd`. Constrói uma régua de **LGD** a partir de uma base sintética, mostra as folhas, a árvore, o `predict` e a régua em PySpark.

**Instalação** (na raiz do repositório):
```bash
pip install -e ".[ui]"   # núcleo + interface interativa
```
> No Databricks: `%pip install -e ".[ui]"` (ou `%pip install optbinning`) num cluster interativo DBR 13.0+ LTS.

In [1]:
import numpy as np
import pandas as pd
from yggdrasil.credit_risk.lgd import SequentialLGDSegmenter

# --- base sintética: LGD de financiamento de veículo, com DES e OOT ---
rng = np.random.default_rng(42)
def gera(n, drift=False):
    ltv = rng.beta(2.5, 3, n) * 1.4 + 0.3
    ltv[rng.random(n) < 0.06] = np.nan          # ~6% de LTV faltante
    atraso = rng.integers(30, 720, n)
    gar = rng.choice(['alienação','aval','fiança','sem garantia'], n, p=[.5,.22,.18,.1])
    lg = {'alienação':0.0,'aval':0.10,'fiança':0.14,'sem garantia':0.28}
    base = (0.05 + 0.45*np.clip(np.nan_to_num(ltv-0.5, nan=0.35), 0, 1)
            + 0.10*(atraso>360) + np.array([lg[g] for g in gar]))
    if drift: base += 0.12*(np.nan_to_num(ltv, nan=0.5) > 0.95)
    return pd.DataFrame({'ltv': ltv, 'dias_atraso': atraso, 'garantia': gar,
                         'lgd': np.clip(base + rng.normal(0, 0.07, n), 0, 1)})
des = gera(6000); des['amostra'] = 'DES'
oot = gera(2500, drift=True); oot['amostra'] = 'OOT'
df = pd.concat([des, oot], ignore_index=True)
df.head()

,ltv,dias_atraso,garantia,lgd,amostra
0,0.850093,245,alienação,0.052402,DES
1,0.960940,690,aval,0.451580,DES
2,1.103200,185,alienação,0.250222,DES
3,0.956844,271,alienação,0.177865,DES
4,1.155575,636,aval,0.499158,DES


## Instanciar e construir a árvore automática

In [2]:
seg = SequentialLGDSegmenter(
    df, target='lgd', sample_col='amostra', ref_sample='DES',
    feature_labels={'ltv':'LTV','dias_atraso':'dias em atraso','garantia':'garantia'},
)
seg.fit_auto(max_depth=3, min_iv=0.02)
seg.suggest_split('root')['msg']

[init] amostras: ['DES', 'OOT'] | referência PSI = DES


/usr/local/lib/python3.12/dist-packages/optbinning/binning/prebinning.py:116: RuntimeWarning: divide by zero encountered in scalar divide
  0.5, self.min_bin_size / np.sum(sample_weight)


/usr/local/lib/python3.12/dist-packages/optbinning/binning/prebinning.py:116: RuntimeWarning: divide by zero encountered in scalar divide
  0.5, self.min_bin_size / np.sum(sample_weight)
/usr/local/lib/python3.12/dist-packages/optbinning/binning/prebinning.py:116: RuntimeWarning: divide by zero encountered in scalar divide
  0.5, self.min_bin_size / np.sum(sample_weight)


[fit_auto] árvore gulosa construída: profundidade 3 (máx 3), IV mínimo 0.02 → 12 folhas


"dividir por 'ltv' (IV=1.1927, suspeito)"

## Folhas finais (nota, LGD, representatividade e PSI)

In [3]:
seg.leaves(with_psi=True)

,segmento,nota_lgd,descricao,profundidade,n,repr_%,lgd_medio,lgd_std,lgd_DES,lgd_OOT,psi_OOT
0,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",1,LTV até 1.35 e garantia em {alienação ou aval ...,3,459,5.4,0.1580,0.0936,0.1532,0.1705,0.0003
1,"ltv: (faltante) | garantia: {alienação, aval, ...",2,LTV faltante e garantia em {alienação ou aval ...,3,243,2.9,0.2629,0.0864,0.2602,0.2684,0.0006
2,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,LTV até 1.35 e garantia em {alienação ou aval ...,3,6230,73.3,0.3650,0.1560,0.3481,0.4056,0.0000
3,"ltv: (faltante) | garantia: {alienação, aval, ...",4,LTV faltante e garantia em {alienação ou aval ...,3,220,2.6,0.3712,0.0904,0.3722,0.3691,0.0003
4,ltv: (faltante) | garantia: {sem garantia} | d...,5,LTV faltante e garantia em {sem garantia} e di...,3,32,0.4,0.5103,0.0652,0.5138,0.5014,0.0000
5,"ltv: (-inf, 1.35] | garantia: {sem garantia} |...",6,LTV até 1.35 e garantia em {sem garantia} e di...,3,371,4.4,0.5171,0.1368,0.5041,0.5524,0.0006
6,"ltv: (1.35, inf] | garantia: {alienação, aval,...",7,LTV acima de 1.35 e garantia em {alienação ou ...,3,266,3.1,0.5627,0.0996,0.5281,0.6377,0.0003
7,ltv: (faltante) | garantia: {sem garantia} | d...,8,LTV faltante e garantia em {sem garantia} e di...,3,18,0.2,0.6078,0.0843,0.5919,0.6878,0.0010
8,"ltv: (-inf, 1.35] | garantia: {sem garantia} |...",9,LTV até 1.35 e garantia em {sem garantia} e di...,3,366,4.3,0.6195,0.1422,0.6104,0.6417,0.0000
9,"ltv: (1.35, inf] | garantia: {alienação, aval,...",10,LTV acima de 1.35 e garantia em {alienação ou ...,3,237,2.8,0.6724,0.1040,0.6364,0.7677,0.0003


## Árvore em texto

In [4]:
print(seg.tree())

TODA A CARTEIRA  (n=8500, 100.0%, LGD=0.3877)
├─ LTV faltante  (n=513, 6.0%, LGD=0.3369)
│  ├─ garantia em {alienação ou aval ou fiança}  (n=463, 5.4%, LGD=0.3144)
│  │  ├─ garantia em {alienação}  (n=243, 2.9%, LGD=0.2629)  [nota 2]
│  │  └─ garantia em {aval ou fiança}  (n=220, 2.6%, LGD=0.3712)  [nota 4]
│  └─ garantia em {sem garantia}  (n=50, 0.6%, LGD=0.5454)
│     ├─ dias em atraso até 455.5  (n=32, 0.4%, LGD=0.5103)  [nota 5]
│     └─ dias em atraso acima de 455.5  (n=18, 0.2%, LGD=0.6078)  [nota 8]
├─ LTV até 1.35  (n=7426, 87.4%, LGD=0.3724)
│  ├─ garantia em {alienação ou aval ou fiança}  (n=6689, 78.7%, LGD=0.3508)
│  │  ├─ LTV até 0.5278  (n=459, 5.4%, LGD=0.1580)  [nota 1]
│  │  └─ LTV acima de 0.5278  (n=6230, 73.3%, LGD=0.3650)  [nota 3]
│  └─ garantia em {sem garantia}  (n=737, 8.7%, LGD=0.5680)
│     ├─ dias em atraso até 347.5  (n=371, 4.4%, LGD=0.5171)  [nota 6]
│     └─ dias em atraso acima de 347.5  (n=366, 4.3%, LGD=0.6195)  [nota 9]
└─ LTV acima de 1.35  (n=561,

## Aplicar a régua em pandas (`predict`)
Repare que linhas com `ltv` faltante são roteadas para o segmento `(faltante)` — nada é descartado.

In [5]:
novos = gera(800)
pred = seg.predict(novos)
print('cobertura (linhas roteadas):', pred['segmento_lgd'].notna().mean())
pred.head()

cobertura (linhas roteadas): 1.0


,segmento_lgd,nota_lgd,lgd_regua
0,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
1,"ltv: (1.35, inf] | garantia: {alienação, aval,...",10,0.636384
2,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
3,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
4,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052


## Régua em PySpark (scoring em escala no Databricks)

In [6]:
print(seg.to_pyspark())

from pyspark.sql import functions as F

def aplicar_regua_lgd(df, col_seg='segmento_lgd', col_nota='nota_lgd', col_lgd='lgd_regua'):
    """Régua de LGD gerada por SequentialLGDSegmenter (segmento, nota e LGD por folha)."""
    c1 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('alienação', 'aval', 'fiança') & (F.col("ltv") <= 0.5277718603610992)
    c2 = F.col("ltv").isNull() & F.col("garantia").isin('alienação', 'aval', 'fiança') & F.col("garantia").isin('alienação')
    c3 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('alienação', 'aval', 'fiança') & (F.col("ltv") > 0.5277718603610992)
    c4 = F.col("ltv").isNull() & F.col("garantia").isin('alienação', 'aval', 'fiança') & F.col("garantia").isin('aval', 'fiança')
    c5 = F.col("ltv").isNull() & F.col("garantia").isin('sem garantia') & (F.col("dias_atraso") <= 455.5)
    c6 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('sem garantia') & (F.col("dias_atraso") <= 347.5)
    c7 = (F.c

## Interface interativa (opcional)
No Jupyter/Databricks, dá para construir a árvore clicando — incluindo seletores de grupo por categoria e nó automático de faltantes:
```python
from yggdrasil.credit_risk.lgd import LGDSegmenterUI
ui = LGDSegmenterUI(df, target='lgd', sample_col='amostra', ref_sample='DES',
                    feature_labels={'ltv':'LTV','dias_atraso':'dias em atraso','garantia':'garantia'})
ui
```

## Registrar no MLflow / Unity Catalog
```python
seg.log_to_mlflow(registered_model_name='catalogo.schema.lgd_segmentacao',
                  registry_uri='databricks-uc')
```